# ML-08 — Capstone Modeling Lane

**Lane:** Refresh / Content Opportunity Scoring. This notebook compares readable learned rankings with the frozen Week-4 baseline on the same client-grouped holdout and the same Precision@K metrics.


## 1. Method choice and why

The practical question is **which pages should be reviewed first?**, so model probabilities are used as ranking scores. I compare Logistic Regression, a small Decision Tree, and Random Forest. All use the same five pre-decision features only; no future-window, label-derived, or existing FlyRank flag is allowed as an input.


In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os, sys, json
from pathlib import Path
import pandas as pd, numpy as np
from google.colab import userdata
from huggingface_hub import whoami
REPO_DIR=Path('/content/Internship')
if not REPO_DIR.exists():
    !git clone https://github.com/imalik-7/Internship.git /content/Internship
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR))
from work.lib.capstone_pipeline import FEATURES,LABEL,CONTEXT,connect_warehouse,build_analysis_frame,grouped_split,train_compare,best_model_name,feature_importance,ensure_output_dir
HF_TOKEN=userdata.get('HF_TOKEN')
print('HF account:',whoami(token=HF_TOKEN)['name'])
con=connect_warehouse(HF_TOKEN)
analysis_df=build_analysis_frame(con)
print('Rows:',len(analysis_df),'| base rate:',round(analysis_df[LABEL].mean(),3))
print('Features:',FEATURES)
assert len(FEATURES)==5


Cloning into '/content/Internship'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 190 (delta 79), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 1.91 MiB | 11.28 MiB/s, done.
Resolving deltas: 100% (79/79), done.
HF account: imalik7


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 77540 | base rate: 0.327
Features: ['impressions_feature15', 'clicks_feature15', 'ctr_feature15', 'avg_position_feature15', 'active_impression_days_feature15']


## 2. Split design

I use a fixed-seed **client-grouped 75/25 holdout**. A client appears in only one side of the split, which makes the test closer to the question: does the ranking generalize to clients it did not see during fitting? The label uses March 16–30 while every feature uses March 1–15.


In [2]:
train_df,test_df=grouped_split(analysis_df,test_size=0.25,random_state=42)
overlap=set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print('Train rows:',len(train_df),'Test rows:',len(test_df))
print('Train clients:',train_df['client_hash_id'].nunique(),'Test clients:',test_df['client_hash_id'].nunique())
print('Client overlap:',len(overlap))
print('Train base rate:',round(train_df[LABEL].mean(),3),'Test base rate:',round(test_df[LABEL].mean(),3))
assert len(overlap)==0


Train rows: 61747 Test rows: 15793
Train clients: 28 Test clients: 10
Client overlap: 0
Train base rate: 0.365 Test base rate: 0.179


## 3. Train + compare vs my baseline

The table below is the required same-data, same-split, same-metric comparison. I report Precision@10, @20, @50 and Average Precision next to the held-out base rate. A learned method only earns its place where it beats the transparent rule under this split.


In [3]:
comparison,fitted_models,test_scores,baseline_test=train_compare(train_df,test_df,random_state=42)
BASE_RATE=float(test_df[LABEL].mean())
BEST_MODEL=best_model_name(comparison)
print('Held-out base rate:',round(BASE_RATE,3))
display(comparison.round(3))
print('Best learned model:',BEST_MODEL)
OUT=ensure_output_dir(REPO_DIR)
with open(OUT/'w05_model_metrics.json','w',encoding='utf-8') as f:
    json.dump({'seed':42,'split':'client-grouped','test_base_rate':BASE_RATE,'best_model':BEST_MODEL,'comparison':comparison.to_dict(orient='records')},f,indent=2)
print('Wrote',OUT/'w05_model_metrics.json')


Held-out base rate: 0.179


,method,precision_at_10,precision_at_20,precision_at_50,average_precision
0,Logistic Regression,0.0,0.20,0.30,0.238
1,Random Forest,0.3,0.30,0.22,0.197
2,Week-4 baseline,0.2,0.15,0.18,0.186
3,Decision Tree,0.2,0.10,0.12,0.196


Best learned model: Logistic Regression
Wrote /content/Internship/work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

A metric without error analysis is decoration. I inspect what the best model leans on and three high-confidence mistakes. Wrong cases may reflect seasonality, query mix, SERP changes, short-window noise, or client context intentionally absent from this public-safe frame.


In [4]:
imp=feature_importance(fitted_models[BEST_MODEL])
display(imp)
ev=test_df[CONTEXT+FEATURES+[LABEL]].copy(); ev['score']=test_scores[BEST_MODEL]; ev['pred']=(ev['score']>=0.5).astype(int); ev['wrong']=ev['pred']!=ev[LABEL]
ev['confidence']=np.where(ev['pred'].eq(1),ev['score'],1-ev['score'])
wrong=ev[ev['wrong']].sort_values('confidence',ascending=False).head(3)
display(wrong[['content_hash_id']+FEATURES+[LABEL,'score','pred']])
print('Interpretation: these are decision-support errors, not proof a page should or should not be changed.')


,feature,importance
0,clicks_feature15,0.431912
1,active_impression_days_feature15,0.346658
2,ctr_feature15,0.237894
3,impressions_feature15,0.219717
4,avg_position_feature15,0.073244


,content_hash_id,impressions_feature15,clicks_feature15,ctr_feature15,avg_position_feature15,active_impression_days_feature15,is_declining_proxy,score,pred
54771,content_747ff6b150f08f84,18917.0,217.0,1.147116,2.363905,13,1,0.002878,0
38793,content_7c47e5b8341f5d53,930.0,52.0,5.591398,2.756989,15,1,0.010601,0
62375,content_8ffa6201737fd638,106.0,8.0,7.547170,4.018868,15,1,0.011718,0


Interpretation: these are decision-support errors, not proof a page should or should not be changed.


## Self-check

- [x] Method fits a ranking/scoring question.
- [x] Five pre-decision features only.
- [x] Client-grouped holdout with no overlap.
- [x] Baseline and models use the same test rows and metrics.
- [x] Base rate, Precision@K, feature importance, and wrong cases are visible after Run all.
- [x] Claims remain observational / directional / decision-support.
